# `mutate` — Reference

`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.

### Calling Styles

1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Spec string / file** (for CLI & config files): `.pt.mutate("bmi = body_mass_g / bill_length_mm ** 2")` or `@specs.txt`

---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2")
    .pt.select("species", "body_mass_g", "bill_length_mm", "bmi")
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
27,Adelie,3200.0,40.5,1.950922
30,Adelie,3250.0,39.5,2.083000
24,Adelie,3800.0,38.8,2.524179
335,Gentoo,5850.0,55.1,1.926871
325,Gentoo,5500.0,46.8,2.511140
96,Adelie,3700.0,38.1,2.548894
315,Gentoo,5200.0,50.8,2.015004
128,Adelie,3050.0,39.0,2.005260
19,Adelie,4200.0,46.0,1.984877
78,Adelie,3550.0,36.2,2.709014


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins.pt.mutate(heavy="body_mass_g > 4000", mass_kg="body_mass_g / 1000")
    .pt.select("species", "body_mass_g", "mass_kg", "heavy")
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
63,Adelie,4050.0,4.050,True
178,Chinstrap,3400.0,3.400,False
187,Chinstrap,3900.0,3.900,False
204,Chinstrap,3600.0,3.600,False
210,Chinstrap,3800.0,3.800,False
100,Adelie,3725.0,3.725,False
238,Gentoo,4800.0,4.800,True
259,Gentoo,5350.0,5.350,True
26,Adelie,3550.0,3.550,False
234,Gentoo,4200.0,4.200,True


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins.pt.mutate(mass_kg="body_mass_g / 1000", mass_lb="mass_kg * 2.20462")
    .pt.select("species", "body_mass_g", "mass_kg", "mass_lb")
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
243,Gentoo,5050.0,5.050,11.133331
290,Gentoo,4750.0,4.750,10.471945
302,Gentoo,4725.0,4.725,10.416829
33,Adelie,3900.0,3.900,8.598018
150,Adelie,3700.0,3.700,8.157094
41,Adelie,3900.0,3.900,8.598018
45,Adelie,4600.0,4.600,10.141252
176,Chinstrap,3300.0,3.300,7.275246
311,Gentoo,5400.0,5.400,11.904948
130,Adelie,3325.0,3.325,7.330361


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins.pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select("species", "is_adelie")
    .sample(10)
)


,species,is_adelie
297,Gentoo,False
340,Gentoo,False
334,Gentoo,False
160,Chinstrap,False
127,Adelie,True
46,Adelie,True
286,Gentoo,False
310,Gentoo,False
43,Adelie,True
250,Gentoo,False


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins.pt.mutate(heavy="body_mass_g >= @threshold")
    .pt.select("species", "body_mass_g", "heavy")
    .sample(10)
)


,species,body_mass_g,heavy
175,Chinstrap,3800.0,False
196,Chinstrap,3675.0,False
39,Adelie,4650.0,True
36,Adelie,3950.0,False
54,Adelie,2900.0,False
213,Chinstrap,3650.0,False
56,Adelie,3550.0,False
246,Gentoo,4100.0,True
110,Adelie,3825.0,False
50,Adelie,3500.0,False


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins.pt.mutate(body_mass_g="body_mass_g / 1000")
    .pt.select("species", "body_mass_g")
    .sample(10)
)


,species,body_mass_g
310,Gentoo,4.950
103,Adelie,4.250
196,Chinstrap,3.675
242,Gentoo,4.400
54,Adelie,2.900
47,Adelie,2.975
316,Gentoo,4.925
254,Gentoo,5.150
267,Gentoo,5.400
25,Adelie,3.800


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2")
    .pt.qry(bmi="> 2")
    .pt.select("species", "island", "body_mass_g", "bill_length_mm", "bmi")
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
81,Adelie,Torgersen,4700.0,42.9,2.553779
91,Adelie,Dream,4300.0,41.1,2.545569
280,Gentoo,Biscoe,4200.0,45.3,2.046694
260,Gentoo,Biscoe,3950.0,42.7,2.166413
56,Adelie,Biscoe,3550.0,39.0,2.333991
287,Gentoo,Biscoe,5800.0,49.5,2.367105
46,Adelie,Dream,3425.0,41.1,2.027575
334,Gentoo,Biscoe,4375.0,46.2,2.049718
92,Adelie,Dream,3400.0,34.0,2.941176
51,Adelie,Biscoe,4300.0,40.1,2.674113


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins.pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select("species", "body_mass_g", "weight_class")
    .sample(10)
)


,species,body_mass_g,weight_class
178,Chinstrap,3400.0,light
118,Adelie,3350.0,light
107,Adelie,3900.0,light
4,Adelie,3450.0,light
80,Adelie,3200.0,light
169,Chinstrap,3700.0,light
224,Gentoo,5400.0,heavy
198,Chinstrap,3400.0,light
171,Chinstrap,4400.0,heavy
267,Gentoo,5400.0,heavy


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins.pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select("species", "body_mass_g", "size_class")
    .sample(10)
)


,species,body_mass_g,size_class
333,Gentoo,5500.0,large
118,Adelie,3350.0,small
106,Adelie,3750.0,medium
112,Adelie,3200.0,small
135,Adelie,3900.0,medium
53,Adelie,4050.0,medium
287,Gentoo,5800.0,large
140,Adelie,3400.0,small
213,Chinstrap,3650.0,medium
30,Adelie,3250.0,small


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins.pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select("species", "island", "island_code")
    .sample(10)
)


,species,island,island_code
173,Chinstrap,Dream,OTH
126,Adelie,Torgersen,TOR
237,Gentoo,Biscoe,BIS
201,Chinstrap,Dream,OTH
339,Gentoo,Biscoe,BIS
81,Adelie,Torgersen,TOR
181,Chinstrap,Dream,OTH
295,Gentoo,Biscoe,BIS
84,Adelie,Dream,OTH
189,Chinstrap,Dream,OTH


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(penguins
 .pt.mutate(
     bmi="body_mass_g / bill_length_mm ** 2",
     mass_kg="body_mass_g / 1000",
     is_heavy=lambda df: df['body_mass_g'] > 4000
 )
 .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
 .head(5)
)

,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`

For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from a file with `#` comments and multiline expressions:

In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = result.pt.select("species", "mass_kg", "mass_lb").sample(5)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out

,species,mass_kg,mass_lb
274,Gentoo,4.900,10.802638
137,Adelie,3.975,8.763364
307,Gentoo,5.300,11.684486
155,Chinstrap,3.525,7.771285
103,Adelie,4.250,9.369635


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581
